In [ ]:
# 1. Install necessary libraries
!pip install opendatasets librosa numpy scikit-learn

import opendatasets as od
import os

# 2. Download GTZAN Dataset
# Link: https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification
od.download("https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: arniuuu
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification


100%|██████████| 1.21G/1.21G [00:12<00:00, 103MB/s]


In [ ]:
import librosa
import numpy as np
from tqdm import tqdm

# Configuration
DATA_PATH = "gtzan-dataset-music-genre-classification/Data/genres_original"
SAMPLE_RATE = 22050
DURATION = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION
NUM_SEGMENTS = 10 # Each 30s track becomes ten 3s samples

def extract_features(data_path):
    X = []
    y = []

    # Get list of genres (folders)
    genres = [g for g in os.listdir(data_path) if not g.startswith('.')]

    samples_per_segment = int(SAMPLES_PER_TRACK / NUM_SEGMENTS)

    for i, genre in enumerate(genres):
        genre_folder = os.path.join(data_path, genre)
        print(f"Processing Genre: {genre}...")

        for f in tqdm(os.listdir(genre_folder)):
            # SKIP CORRUPTED FILE
            if f == "jazz.00054.wav":
                continue

            file_path = os.path.join(genre_folder, f)
            try:
                # Load audio file
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

                # Process segments
                for d in range(NUM_SEGMENTS):
                    start = samples_per_segment * d
                    finish = start + samples_per_segment

                    # Extract MFCCs (13 coefficients)
                    mfcc = librosa.feature.mfcc(y=signal[start:finish], sr=sr, n_mfcc=13, n_fft=2048, hop_length=512)

                    # We take the mean across time so we have a 1D vector per segment
                    mfcc_scaled = np.mean(mfcc.T, axis=0)

                    X.append(mfcc_scaled)
                    y.append(genre)

            except Exception as e:
                print(f"Error loading {f}: {e}")

    return np.array(X), np.array(y)

print("Functions defined successfully.")

Functions defined successfully.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Run the extraction
print("Starting extraction (this may take 5-10 minutes)...")
X, y = extract_features(DATA_PATH)

# 2. Encode labels (Text -> Numbers)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Scale the features (Mean=0, Variance=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

print(f"\n✅ Data Ready!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Genres found: {le.classes_}")

Starting extraction (this may take 5-10 minutes)...
Processing Genre: pop...


100%|██████████| 100/100 [00:27<00:00,  3.69it/s]


Processing Genre: disco...


100%|██████████| 100/100 [00:07<00:00, 12.83it/s]


Processing Genre: reggae...


...

Processing Genre: country...


100%|██████████| 100/100 [00:08<00:00, 11.62it/s]


Processing Genre: classical...


100%|██████████| 100/100 [00:09<00:00, 10.59it/s]


✅ Data Ready!
Training samples: 7992
Testing samples: 1998
Genres found: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']


In [ ]:
# Save binary files
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)
np.save("genre_classes.npy", le.classes_)

print("Files saved: X_train.npy, X_test.npy, y_train.npy, y_test.npy, genre_classes.npy")
print("Member 1: Download these from the folder icon on the left!")

Files saved: X_train.npy, X_test.npy, y_train.npy, y_test.npy, genre_classes.npy
Member 1: Download these from the folder icon on the left!


In [ ]:
import librosa
import numpy as np
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# 1. Extraction Function for 2D Spectrograms
def extract_spectrogram_images(data_path):
    X = []
    y = []
    genres = [g for g in os.listdir(data_path) if not g.startswith('.')]
    samples_per_segment = int(SAMPLE_RATE * 30 / NUM_SEGMENTS)

    for i, genre in enumerate(genres):
        genre_folder = os.path.join(data_path, genre)
        print(f"Creating 2D Spectrograms for: {genre}...")

        for f in tqdm(os.listdir(genre_folder)):
            if f == "jazz.00054.wav": continue

            file_path = os.path.join(genre_folder, f)
            try:
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                for d in range(NUM_SEGMENTS):
                    start, finish = samples_per_segment * d, samples_per_segment * (d + 1)

                    # Generate Mel Spectrogram
                    mel_spec = librosa.feature.melspectrogram(y=signal[start:finish], sr=sr, n_mels=128)
                    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

                    # --- FIX: Ensure exact 128x128 shape ---
                    if log_mel_spec.shape[1] >= 128:
                        log_mel_spec = log_mel_spec[:, :128]
                    else:
                        log_mel_spec = np.pad(log_mel_spec, ((0,0), (0, 128 - log_mel_spec.shape[1])), mode='constant')

                    X.append(log_mel_spec)
                    y.append(i)
            except Exception: continue

    return np.array(X), np.array(y)

# 2. Execute and Reshape for CNN
X_img, y_img = extract_spectrogram_images(DATA_PATH)
X_img = X_img[..., np.newaxis] # Adds the 'channel' dimension (Samples, 128, 128, 1)

# 3. Split
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_img, y_img, test_size=0.2, random_state=42)

# 4. Save Image-Specific Files
np.save("X_train_images.npy", X_train_img)
np.save("X_test_images.npy", X_test_img)
np.save("y_train_images.npy", y_train_img)
np.save("y_test_images.npy", y_test_img)

print("\n✅ MISSION COMPLETE: Image-based dataset saved!")
print(f"Final Image Shape: {X_train_img.shape[1:]}")

Creating 2D Spectrograms for: pop...


100%|██████████| 100/100 [00:09<00:00, 10.64it/s]


Creating 2D Spectrograms for: disco...


100%|██████████| 100/100 [00:08<00:00, 12.23it/s]


Creating 2D Spectrograms for: reggae...


...

Creating 2D Spectrograms for: country...


100%|██████████| 100/100 [00:09<00:00, 10.29it/s]


Creating 2D Spectrograms for: classical...


100%|██████████| 100/100 [00:09<00:00, 10.20it/s]



✅ MISSION COMPLETE: Image-based dataset saved!
Final Image Shape: (128, 128, 1)
